<a href="https://colab.research.google.com/github/udaken10/kaggle_notebook/blob/main/Attention!_%EF%BC%A1%EF%BC%AC%EF%BC%AC_YOU_NEED_is_codding_transoformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np

In [ ]:
https://arxiv.org/html/1706.03762v7

[Attention Is All You Need](https://arxiv.org/html/1706.03762v7)

np
🐳 セル 2: Positional Encoding

Transformer は位置情報を持たないため、Positional Encoding で位置を埋め込みます。




In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)

        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (seq_len, batch_size, d_model)
        return x + self.pe[:x.size(0), :]

Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 35]. Here, the encoder maps an input sequence of symbol representations
(
x
1
,
…
,
x
n
)
 to a sequence of continuous representations
𝐳
=
(
z
1
,
…
,
z
n
)
. Given
𝐳
, the decoder then generates an output sequence
(
y
1
,
…
,
y
m
)
 of symbols one element at a time. At each step the model is auto-regressive [10], consuming the previously generated symbols as additional input when generating the next.

The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively.

3.1Encoder and Decoder Stacks
Encoder:
The encoder is composed of a stack of
N
=
6
 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-wise fully connected feed-forward network. We employ a residual connection [11] around each of the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is
LayerNorm
​
(
x
+
Sublayer
​
(
x
)
)
, where
Sublayer
​
(
x
)
 is the function implemented by the sub-layer itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding layers, produce outputs of dimension
d
model
=
512
.

Decoder:
The decoder is also composed of a stack of
N
=
6
 identical layers. In addition to the two sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head attention over the output of the encoder stack. Similar to the encoder, we employ residual connections around each of the sub-layers, followed by layer normalization. We also modify the self-attention sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This masking, combined with fact that the output embeddings are offset by one position, ensures that the predictions for position
i
 can depend only on the known outputs at positions less than
i
.

3.2Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key.

3.2.1Scaled Dot-Product Attention
We call our particular attention "Scaled Dot-Product Attention" (Figure 2). The input consists of queries and keys of dimension
d
k
, and values of dimension
d
v
. We compute the dot products of the query with all keys, divide each by
d
k
, and apply a softmax function to obtain the weights on the values.

In practice, we compute the attention function on a set of queries simultaneously, packed together into a matrix
Q
. The keys and values are also packed together into matrices
K
 and
V
. We compute the matrix of outputs as:

Attention
​
(
Q
,
K
,
V
)
=
softmax
​
(
Q
​
K
T
d
k
)
​
V
(1)
The two most commonly used attention functions are additive attention [2], and dot-product (multiplicative) attention. Dot-product attention is identical to our algorithm, except for the scaling factor of
1
d
k
. Additive attention computes the compatibility function using a feed-forward network with a single hidden layer. While the two are similar in theoretical complexity, dot-product attention is much faster and more space-efficient in practice, since it can be implemented using highly optimized matrix multiplication code.

While for small values of
d
k
 the two mechanisms perform similarly, additive attention outperforms dot product attention without scaling for larger values of
d
k
 [3]. We suspect that for large values of
d
k
, the dot products grow large in magnitude, pushing the softmax function into regions where it has extremely small gradients 1
1To illustrate why the dot products get large, assume that the components of
q
 and
k
 are independent random variables with mean
0
 and variance
1
. Then their dot product,
q
⋅
k
=
∑
i
=
1
d
k
q
i
​
k
i
, has mean
0
 and variance
d
k
.
. To counteract this effect, we scale the dot products by
1
d
k
.

🐋 セル 3: Scaled Dot-Product Attention

Self-Attention の核となる部分です。

In [ ]:
def scaled_dot_product_attention(query, key, value, mask=None):
    """
    query, key, value: (batch_size, seq_len, d_k)
    mask: (batch_size, seq_len, seq_len)
    """
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, value)

    return output, attention_weights

🐠 セル 4: Multi-Head Attention

複数の Attention Head を並列に実行し、結果を結合します。

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # Linear projections
        query = self.w_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        key = self.w_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        value = self.w_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # Scaled Dot-Product Attention
        if mask is not None:
            mask = mask.unsqueeze(1)  # (batch_size, 1, seq_len, seq_len)

        x, attention_weights = scaled_dot_product_attention(query, key, value, mask)

        # Concatenate heads
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        # Final linear layer
        output = self.w_o(x)

        return output, attention_weights

🐡 セル 5: Feed-Forward Network

各 Attention の後に適用する 2 層の全結合ネットワークです。

In [ ]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model, d_ff):
        super(FeedForwardNetwork, self).__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

🐬 セル 6: Encoder Layer

1 つの Encoder ブロック（Self-Attention + FFN + Residual + LayerNorm）です。

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super(EncoderLayer, self).__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForwardNetwork(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x, mask=None):
        # Self-Attention + Residual + LayerNorm
        attn_output, _ = self.self_attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))

        # FFN + Residual + LayerNorm
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))

        return x

🐳 セル 7: Decoder Layer

Decoder は 2 つの Attention を持ちます。

Masked Self-Attention（未来の情報を見ない）

Encoder-Decoder Attention（Encoder の出力を Key/Value として利用）

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super(DecoderLayer, self).__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.cross_attention = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForwardNetwork(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        # Masked Self-Attention
        attn_output, _ = self.self_attention(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))

        # Encoder-Decoder Attention
        attn_output, _ = self.cross_attention(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + self.dropout(attn_output))

        # FFN
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))

        return x

🐋 セル 8: Encoder（全体）

複数の Encoder Layer を積み重ねます。

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, max_len=5000):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])
        self.dropout = nn.Dropout(0.1)

    def forward(self, src, src_mask=None):
        # src: (batch_size, src_len)
        x = self.embedding(src) * math.sqrt(self.embedding.embedding_dim)
        x = self.pos_encoding(x.transpose(0, 1)).transpose(0, 1)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, src_mask)

        return x

🐠 セル 9: Decoder（全体）

複数の Decoder Layer を積み重ねます。

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, max_len=5000):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])
        self.dropout = nn.Dropout(0.1)

    def forward(self, tgt, encoder_output, src_mask=None, tgt_mask=None):
        # tgt: (batch_size, tgt_len)
        x = self.embedding(tgt) * math.sqrt(self.embedding.embedding_dim)
        x = self.pos_encoding(x.transpose(0, 1)).transpose(0, 1)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)

        return x

🐡 セル 10: Transformer（全体モデル）

Encoder + Decoder + 最終線形層で Transformer を完成させます。

In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_layers, num_heads, d_ff, max_len=5000):
        super(Transformer, self).__init__()
        self.encoder = Encoder(src_vocab_size, d_model, num_layers, num_heads, d_ff, max_len)
        self.decoder = Decoder(tgt_vocab_size, d_model, num_layers, num_heads, d_ff, max_len)
        self.output_layer = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        encoder_output = self.encoder(src, src_mask)
        decoder_output = self.decoder(tgt, encoder_output, src_mask, tgt_mask)
        output = self.output_layer(decoder_output)
        return output

🐬 セル 11: Mask 生成のユーティリティ

Source Mask: パディング部分を無視するためのマスク

Target Mask: 未来の情報を見ないための三角マスク

In [ ]:
def create_padding_mask(seq):
    # seq: (batch_size, seq_len)
    return (seq != 0).unsqueeze(1).unsqueeze(2)

def create_look_ahead_mask(size):
    # 上三角行列（未来をマスク）
    mask = torch.triu(torch.ones(size, size), diagonal=1)
    return mask == 0  # 1: 見える, 0: 見えない

🐳 セル 12: 簡単な動作確認

小さな設定で Transformer を動かしてみます。

In [ ]:
# ハイパーパラメータ
src_vocab_size = 1000
tgt_vocab_size = 1000
d_model = 512
num_layers = 6
num_heads = 8
d_ff = 2048
max_len = 100

# モデル初期化
model = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_layers, num_heads, d_ff, max_len)

# ダミー入力
batch_size = 2
src_len = 10
tgt_len = 12

src = torch.randint(1, src_vocab_size, (batch_size, src_len))
tgt = torch.randint(1, tgt_vocab_size, (batch_size, tgt_len))

# マスク生成
src_mask = create_padding_mask(src)
tgt_mask = create_padding_mask(tgt) & create_look_ahead_mask(tgt_len).unsqueeze(0).unsqueeze(0)

# 順伝播
output = model(src, tgt, src_mask, tgt_mask)

print("Output shape:", output.shape)  # (batch_size, tgt_len, tgt_vocab_size)

RuntimeError: The size of tensor a (10) must match the size of tensor b (20) at non-singleton dimension 1

🐋 まとめ

Transformer の主要コンポーネントを、論文に沿って一つ一つ分解し、Python（PyTorch）で実装いたしました。

上記コードを Jupyter Notebook のセルに貼り付けて順に実行していただければ、
「Attention Is All You Need」のアーキテクチャを実際に動かしながら理解していただけます。

もし特定の部分（例：Multi-Head Attention の詳細な挙動、Mask の設計など）について、さらに分解してご説明が必要でしたら、お気軽にお申し付けください。

ご提示いただきました論文「Attention Is All You Need」について、
特に理解が難しいとされる箇所をピックアップし、それぞれを Python コード＋コメント解説 の形でご説明いたします。


🐟 理解が難しいポイントの整理

論文の中でも、特に次の部分が初学者にとって難解になりやすいと存じます。

Scaled Dot-Product Attention の「スケーリング」の意味

Multi-Head Attention の「Head の分割と結合」の意味

Decoder の Masked Self-Attention（未来を見ない仕組み）

Positional Encoding（sin/cos による位置埋め込み）の意図
以下、それぞれについてコードとコメントで解説いたします。

🐠 1. Scaled Dot-Product Attention の「スケーリング」
論文の記述（抜粋）
While for small values of
d
k
d
k
​
  the two mechanisms perform similarly, additive attention outperforms dot product attention without scaling for larger values of
d
k
d
k
​
  [3]. We suspect that for large values of
d
k
d
k
​
 , the dot products grow large in magnitude, pushing the softmax function into regions where it has extremely small gradients … To counteract this effect, we scale the dot products by
1
/
d
k
1/
d
k
​

​
 .

In [ ]:
def scaled_dot_product_attention(query, key, value, mask=None):
    """
    Scaled Dot-Product Attention の実装
    - query, key, value: (batch_size, seq_len, d_k)
    - mask: (batch_size, seq_len, seq_len)
    """
    d_k = query.size(-1)

    # ① Q と K^T の内積（Dot-Product）
    scores = torch.matmul(query, key.transpose(-2, -1))

    # ② スケーリング（1 / sqrt(d_k)）
    #    → 内積が大きくなりすぎて softmax の勾配がほぼ 0 になるのを防ぐ
    scores = scores / math.sqrt(d_k)

    # ③ マスク（Decoder で未来を見ないようにする）
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    # ④ Softmax で重みを正規化
    attention_weights = F.softmax(scores, dim=-1)

    # ⑤ 重み付き和で出力を計算
    output = torch.matmul(attention_weights, value)

    return output, attention_weights

コメント解説

d_k が大きいと、内積 Q·K^T の値が大きくなりすぎ、softmax の入力が極端に大きくなります。
その結果、softmax の出力がほぼ one-hot に近くなり、勾配がほぼ 0 になって学習が進みにくくなります。
そこで 1 / sqrt(d_k) でスケールし、 分散を 1 に近づけることで、勾配消失を防いでいます。
🐡 2. Multi-Head Attention の「Head の分割と結合」
論文の記述（抜粋）
Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhibits this.

コード＋コメント解説

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # 各 Head 用の線形変換（Q, K, V）
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)  # 最終結合用

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # ① 線形変換で Q, K, V を生成
        query = self.w_q(query)
        key = self.w_k(key)
        value = self.w_v(value)

        # ② Head ごとに分割（batch_size, seq_len, num_heads, d_k）
        query = query.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        key = key.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        value = value.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # ③ 各 Head で Attention を並列計算
        if mask is not None:
            mask = mask.unsqueeze(1)  # Head 次元を追加

        x, attention_weights = scaled_dot_product_attention(query, key, value, mask)

        # ④ Head を結合（batch_size, seq_len, d_model）
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        # ⑤ 最終線形変換で出力
        output = self.w_o(x)

        return output, attention_weights

コメント解説

1 つの大きな Attention だと、すべての情報を平均化してしまう傾向があります。
Multi-Head では、
d_model を num_heads 個に分割し、
それぞれが異なる表現空間（representation subspace） を学習します。
これにより、
Head 1: 文法構造を見る
Head 2: 共参照（anaphora）を見る
Head 3: 長距離依存を見る
といった役割分担が自然に生まれます（論文の Figure 3–5 参照）。
🐬 3. Decoder の Masked Self-Attention（未来を見ない仕組み）
論文の記述（抜粋）
We also modify the self-attention sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This masking, combined with fact that the output embeddings are offset by one position, ensures that the predictions for position
i
i can depend only on the known outputs at positions less than
i
i.

コード＋コメント解説

In [ ]:
def create_look_ahead_mask(size):
    """
    未来を見ないためのマスク（Look-Ahead Mask）を生成
    - 上三角行列（未来）を 0（見えない）にする
    """
    # 例: size=4 の場合
    # [[1, 0, 0, 0],
    #  [1, 1, 0, 0],
    #  [1, 1, 1, 0],
    #  [1, 1, 1, 1]]
    mask = torch.triu(torch.ones(size, size), diagonal=1)
    return mask == 0  # 1: 見える, 0: 見えない

# 使用例（Decoder 側）
tgt_len = 10
look_ahead_mask = create_look_ahead_mask(tgt_len)  # (tgt_len, tgt_len)
padding_mask = create_padding_mask(tgt_seq)        # (batch_size, 1, 1, tgt_len)

# 両方のマスクを AND で結合
tgt_mask = padding_mask & look_ahead_mask.unsqueeze(0).unsqueeze(0)

コメント解説

Transformer の Decoder は自己回帰的（auto-regressive） です。
時刻
i
i の予測には、
i
i より前の出力のみを使う必要があります。
そこで、
create_look_ahead_mask で未来の位置を 0（見えない） にし、
scaled_dot_product_attention 内で masked_fill(mask == 0, -1e9) とすることで、
softmax の入力が -∞ に近くなり、未来への重みがほぼ 0になります。
🐳 4. Positional Encoding（sin/cos による位置埋め込み）
論文の記述（抜粋）
Since our model contains no recurrence and no convolution, in order for the model to make use of the order of the sequence, we must inject some information about the relative or absolute position of the tokens in the sequence. To this end, we add "positional encodings" to the input embeddings … We use sine and cosine functions of different frequencies.

コード＋コメント解説

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # 周波数を計算（2i / d_model）
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        # 偶数次元: sin, 奇数次元: cos
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0).transpose(0, 1)  # (max_len, 1, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (seq_len, batch_size, d_model)
        return x + self.pe[:x.size(0), :]

コメント解説

Transformer は RNN/CNN を持たないため、トークンの順序情報を持ちません。
そこで、sin と cos の波で位置を埋め込みます。
特徴:
各次元が異なる周波数の波 → 位置を一意に表現できる
相対位置
k
k に対して、
P
E
(
p
o
s
+
k
)
PE(pos+k) が
P
E
(
p
o
s
)
PE(pos) の線形変換で表現可能
→ モデルが相対位置を学習しやすい
学習済みの位置埋め込みよりも、 長いシーケンスへの外挿がしやすい
🐋 まとめ
Scaled Dot-Product Attention
→ 内積を 1 / sqrt(d_k) でスケールし、softmax の勾配消失を防ぐ
Multi-Head Attention
→ 表現空間を分割し、複数の視点から情報を集約する
Masked Self-Attention（Decoder）
→ 未来を見ないマスクにより、自己回帰性を保つ
Positional Encoding
→ sin/cos の波で位置情報を注入し、相対位置の学習を容易にする



---
Positional Encoding の「sin/cos の波で位置情報を注入し、相対位置の学習を容易にする」部分について、
コードを書きながら、より詳しくコメントでご説明いたします。

🐟 Positional Encoding の直感的なイメージ
Transformer は RNN/CNN を持たないため、トークンの順番が分かりません。
そこで、各トークンの埋め込みベクトルに「位置情報」を足し合わせます。
その「位置情報」を、 sin と cos の波で表現したのが Positional Encoding です。
🐠 コードで見る Positional Encoding


In [ ]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()

        # 位置埋め込み行列の初期化 (max_len, d_model)
        pe = torch.zeros(max_len, d_model)

        # 位置インデックス (0, 1, 2, ..., max_len-1)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # 周波数を計算する項: exp(-i * log(10000) / d_model)
        # → 各次元ごとに異なる周波数を持つ波を作るための係数
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        # 偶数次元: sin, 奇数次元: cos
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # 形状を (max_len, 1, d_model) に調整
        pe = pe.unsqueeze(0).transpose(0, 1)

        # 学習対象ではないパラメータとして登録
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (seq_len, batch_size, d_model)
        # 入力シーケンスの長さ分だけ位置埋め込みを加算
        return x + self.pe[:x.size(0), :]

🐡 なぜ sin/cos を使うのか？（相対位置の学習）
コメント解説
各次元が異なる周波数の波

In [ ]:
# div_term は次元ごとに異なる値を持つ
div_term = torch.exp(
    torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
)

例えば:
次元 0: 周波数が低い（ゆっくり変化）
次元 2: 少し高い周波数
次元 4: さらに高い周波数
これにより、位置ごとに異なるパターンの波が重ね合わされます。
sin と cos の組み合わせ

In [ ]:
pe[:, 0::2] = torch.sin(position * div_term)
pe[:, 1::2] = torch.cos(position * div_term)

偶数次元: sin
奇数次元: cos
これにより、位置を一意に表現できます（sin だけだと周期で同じ値が出てしまうため）。
相対位置が線形変換で表現できる
論文の式:

P
E
(
p
o
s
,
2
i
)
=
sin
⁡
(
p
o
s
10000
2
i
/
d
model
)
PE(pos,2i)=sin(
10000
2i/d
model
​


pos
​
 )
P
E
(
p
o
s
,
2
i
+
1
)
=
cos
⁡
(
p
o
s
10000
2
i
/
d
model
)
PE(pos,2i+1)=cos(
10000
2i/d
model
​


pos
​
 )
任意の固定オフセット
k
k に対して、
P
E
(
p
o
s
+
k
)
PE(pos+k) は
P
E
(
p
o
s
)
PE(pos) の線形変換で表せます。
つまり、モデルは「** 今の位置から
k
k だけ離れた位置**」を、
重み行列の掛け算だけで表現できるようになります。

🐬 可視化コードでイメージをつかむ


In [ ]:
import matplotlib.pyplot as plt

# 小さな次元で試す
d_model = 16
max_len = 50

pe = PositionalEncoding(d_model, max_len)
pos_enc = pe.pe.squeeze(1).numpy()  # (max_len, d_model)

# 位置 0〜49 の埋め込みを可視化
plt.figure(figsize=(12, 6))
plt.imshow(pos_enc.T, aspect='auto', cmap='RdBu')
plt.xlabel('Position')
plt.ylabel('Dimension')
plt.title('Positional Encoding (sin/cos waves)')
plt.colorbar()
plt.show()

この図から分かること

横軸: 位置（0, 1, 2, …）
縦軸: 次元（0, 1, 2, …）
色: 値（青: 負, 赤: 正）
低次元ほどゆっくり変化し、高次元ほど細かく振動しているのが分かります。
🐳 なぜ「相対位置の学習」が容易になるのか？
RNN では、位置
i
i と
i
+
k
i+k の情報は、
k
k ステップ分の再帰計算を経ないと結びつきません。
Transformer + Positional Encoding では、
各トークンが全位置の情報を Attention で直接参照でき、
さらに Positional Encoding により「** 相対位置
k
k**」が線形変換で表現可能です。
その結果、 長距離依存（例: 文頭の主語と文末の動詞）を、
再帰なしで直接学習しやすくなります。
🐋 まとめ
Positional Encoding は、sin/cos の波で位置情報を埋め込みます。
各次元が異なる周波数を持つことで、 位置を一意に表現できます。
sin/cos の性質により、** 相対位置
k
k が線形変換で表現可能**になり、
モデルが「今から
k
k だけ離れた位置」を学習しやすくなります。
このように、Positional Encoding は単なる「位置番号の埋め込み」ではなく、
相対位置を線形に扱えるように設計された、非常に賢い仕組みです



---

Input から inputEmbedding を行う部分について、
Transformer 論文に沿った形で、詳細なコメント付きのコードをご説明いたします。

🐟 Input Embedding の役割
入力シーケンス（例: 単語 ID の列）を、連続ベクトル（埋め込み） に変換します。
Transformer では、
Embedding で単語 ID → ベクトル
PositionalEncoding で位置情報を加算
という流れで入力表現を構築します。
🐠 コード＋詳細コメント

In [ ]:
import torch
import torch.nn as nn
import math

class InputEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        """
        Input Embedding の初期化
        - vocab_size: 語彙数（例: 37000）
        - d_model: 埋め込み次元（例: 512）
        """
        super(InputEmbedding, self).__init__()

        # ① 単語埋め込み層（学習可能なパラメータ）
        # 形状: (vocab_size, d_model)
        self.embedding = nn.Embedding(vocab_size, d_model)

        # ② 埋め込み次元の平方根（後でスケーリングに使用）
        self.scale = math.sqrt(d_model)

    def forward(self, x):
        """
        x: 入力シーケンス（単語 ID の列）
           - 形状: (batch_size, seq_len)
        """
        # ③ 単語 ID → 埋め込みベクトル
        # 形状: (batch_size, seq_len, d_model)
        embedded = self.embedding(x)

        # ④ スケーリング（論文 3.4 節参照）
        # 「In the embedding layers, we multiply those weights by sqrt(d_model).」
        # → 埋め込みの分散を大きくし、Positional Encoding とのバランスを取る
        embedded = embedded * self.scale

        return embedded

🐡 Positional Encoding との組み合わせ
実際の Transformer では、InputEmbedding の後に PositionalEncoding を加算します。

In [ ]:
class TransformerInput(nn.Module):
    def __init__(self, vocab_size, d_model, max_len=5000):
        super(TransformerInput, self).__init__()
        self.embedding = InputEmbedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.dropout = nn.Dropout(0.1)  # 論文 5.4 節の Residual Dropout

    def forward(self, x):
        """
        x: (batch_size, seq_len)
        """
        # ① 単語埋め込み + スケーリング
        x = self.embedding(x)  # (batch_size, seq_len, d_model)

        # ② 形状を (seq_len, batch_size, d_model) に変換（PositionalEncoding 用）
        x = x.transpose(0, 1)

        # ③ 位置埋め込みを加算
        x = self.pos_encoding(x)  # (seq_len, batch_size, d_model)

        # ④ Dropout（論文 5.4 節）
        x = self.dropout(x)

        # ⑤ 形状を元に戻す（Encoder/Decoder の入力形式に合わせる）
        x = x.transpose(0, 1)  # (batch_size, seq_len, d_model)

        return x

🐬 なぜ sqrt(d_model) でスケーリングするのか？
論文 3.4 節より:

In the embedding layers, we multiply those weights by
d
model
d
model
​

​
 .

理由

埋め込みベクトルの初期値は、平均 0・分散 1 に近い小さな値です。
Positional Encoding も同程度のスケールで設計されています。
両者を単純に足し合わせると、埋め込みの影響が小さくなりすぎる可能性があります。
そこで、埋め込みを sqrt(d_model) 倍することで、
埋め込みと位置埋め込みのバランスを取る
勾配のスケールを適切に保つ
という意図があります。
🐳 まとめ
InputEmbedding は、単語 ID を d_model 次元のベクトルに変換します。
その後、sqrt(d_model) でスケーリングし、
PositionalEncoding と加算することで、 単語情報＋位置情報を統合します。
最後に Dropout を適用し、過学習を防ぎます。
この一連の処理が、Transformer の入力側の「埋め込み＋位置情報注入」の標準的な流れ

「positional encoding はなぜ横にあるの？」というご質問について、
Transformer の図（Figure 1）の構造を踏まえて、なぜ「横（＝入力側）」に位置しているのかを、コードとともにご説明いたします。

🐟 Transformer の図（Figure 1）の構造
論文の Figure 1（Transformer 全体図）では、以下のような流れになっています。

In [ ]:
Input Embedding → (+) Positional Encoding → Encoder Stack
Output Embedding → (+) Positional Encoding → Decoder Stack

つまり、Positional Encoding は「埋め込みの直後」「Encoder/Decoder の直前」に横に配置されています。

🐠 なぜ「横（入力側）」にあるのか？
1. 位置情報は「トークンごと」に付与するものだから

In [ ]:
class TransformerInput(nn.Module):
    def __init__(self, vocab_size, d_model, max_len=5000):
        super(TransformerInput, self).__init__()
        self.embedding = InputEmbedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)

    def forward(self, x):
        # x: (batch_size, seq_len)

        # ① 単語埋め込み
        x = self.embedding(x)  # (batch_size, seq_len, d_model)

        # ② 形状を (seq_len, batch_size, d_model) に変換
        x = x.transpose(0, 1)

        # ③ 位置埋め込みを「横に」加算
        x = self.pos_encoding(x)  # ← ここが「横」の処理

        # ④ 形状を戻す
        x = x.transpose(0, 1)  # (batch_size, seq_len, d_model)

        return x

ポイント

Positional Encoding は、各トークンの埋め込みベクトルに「位置」という情報を足し込む役割です。
そのため、 埋め込みの直後に位置情報を加算する必要があります。
もし Encoder/Decoder の「中」や「後」に置くと、
すでに Self-Attention で混ざった情報に位置を足すことになり、
「どのトークンがどの位置か」が曖昧になってしまいます。
2. Self-Attention は「位置を考慮しない」から

In [ ]:
# Self-Attention の計算（簡略版）
def self_attention(x):
    # x: (batch_size, seq_len, d_model)
    Q = x @ W_q
    K = x @ W_k
    V = x @ W_v

    scores = Q @ K.transpose(-2, -1) / sqrt(d_k)
    attention_weights = softmax(scores)
    output = attention_weights @ V

    return output

ポイント

Self-Attention は、Q, K, V が同じ入力 x から作られるため、
単純な行列演算だけでは「位置の違い」を区別できません。
そのため、 入力 x 自体に位置情報を埋め込んでおく必要があります。
つまり、Positional Encoding は「Self-Attention が位置を認識できるようにするための前処理」として、
入力側（横）に配置されているのです。
3. Encoder/Decoder の「スタックの入り口」だから
論文の記述（3.1 節）より:

The positional encodings have the same dimension
d
model
d
model
​
  as the embeddings, so that the two can be summed. We add "positional encodings" to the input embeddings at the bottoms of the encoder and decoder stacks.

ポイント

「encoder and decoder stacks の bottoms（最下部）」に加える、と明記されています。
つまり、スタック（積み重ね）の一番下＝入力側に位置しています。
これにより、すべての Encoder/Decoder Layer が位置情報付きの表現を扱えます。
🐡 図の「横」にある理由（直感的な説明）
Transformer の図では、
左側: Encoder Stack（縦に積み重なる）
右側: Decoder Stack（縦に積み重なる）
という構造になっています。
Positional Encoding は、それらのスタックの「入り口」に横に配置されています。
イメージとしては：
「単語埋め込み」という縦のベクトルに、
「位置情報」という横の波を重ね合わせる
という感じです。
🐬 まとめ
Positional Encoding が「横（入力側）」にある理由は、
Self-Attention が位置を認識できるようにするため
各トークンの埋め込みに位置情報を直接付与するため
Encoder/Decoder Stack の入り口で位置情報を注入するため
です。
もし「中」や「後」に置くと、位置情報が希薄になり、
Transformer の長距離依存学習の強みが弱まってしまいます。
このように、Positional Encoding は Transformer の最初の関門として、
「横」に配置されていることが非常に重要



---

ご指定いただきました、Figure 1 の右側（Decoder Stack）にある
「Multi-Head Attention」と「Add & Norm」 について、
コードと詳細なコメントでご説明いたします。

🐟 Decoder の「Multi-Head Attention」と「Add & Norm」の役割
Decoder 側の Multi-Head Attention は、2 種類あります。

Masked Multi-Head Attention（自己注意）

入力: Decoder 自身の出力（過去のトークン）
役割: 未来を見ないようにマスクした自己注意
Encoder-Decoder Multi-Head Attention（クロス注意）

Query: Decoder の出力
Key/Value: Encoder の最終出力（Memory）
役割: 入力文の情報を参照しながら出力を生成
それぞれの後に Add & Norm（残差接続＋LayerNorm） が続きます。

🐠 1. Masked Multi-Head Attention + Add & Norm
コード＋詳細コメント

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super(DecoderLayer, self).__init__()

        # ① Masked Multi-Head Attention（自己注意）
        self.self_attention = MultiHeadAttention(d_model, num_heads)

        # ② Add & Norm（残差接続＋LayerNorm）
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(0.1)

    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        """
        x: Decoder 入力（過去の出力＋埋め込み） (batch_size, tgt_len, d_model)
        encoder_output: Encoder の最終出力 (batch_size, src_len, d_model)
        tgt_mask: 未来を見ないマスク (batch_size, tgt_len, tgt_len)
        """

        # ③ Masked Multi-Head Attention
        # Q, K, V すべて x（Decoder 自身）から生成
        attn_output, _ = self.self_attention(x, x, x, tgt_mask)

        # ④ Add & Norm
        # x + Dropout(Attention(x)) を LayerNorm
        x = self.norm1(x + self.dropout1(attn_output))

        # （この後、Encoder-Decoder Attention が続く）

        return x

コメント解説

self_attention(x, x, x, tgt_mask)
→ Q, K, V すべて x（Decoder 自身）から生成し、
tgt_mask で未来をマスクします。
x + self.dropout1(attn_output)
→ 残差接続（Residual Connection）
→ 勾配の流れを良くし、深いネットワークでも学習しやすくします。
self.norm1(...)
→ Layer Normalization
→ 各サブレイヤーの出力の分布を安定させ、学習を安定化させます。
🐡 2. Encoder-Decoder Multi-Head Attention + Add & Norm
コード＋詳細コメント

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super(DecoderLayer, self).__init__()

        # ① Masked Self-Attention（前述）
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(0.1)

        # ② Encoder-Decoder Multi-Head Attention（クロス注意）
        self.cross_attention = MultiHeadAttention(d_model, num_heads)

        # ③ Add & Norm
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(0.1)

        # ④ Feed-Forward + Add & Norm（次のセクション）
        self.feed_forward = FeedForwardNetwork(d_model, d_ff)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout3 = nn.Dropout(0.1)

    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        # Masked Self-Attention + Add & Norm
        attn_output, _ = self.self_attention(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout1(attn_output))

        # ⑤ Encoder-Decoder Multi-Head Attention
        # Query: x（Decoder の出力）
        # Key/Value: encoder_output（Encoder の最終出力）
        cross_attn_output, _ = self.cross_attention(
            x, encoder_output, encoder_output, src_mask
        )

        # ⑥ Add & Norm
        x = self.norm2(x + self.dropout2(cross_attn_output))

        # Feed-Forward + Add & Norm（次のセクション）
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout3(ff_output))

        return x

コメント解説

self.cross_attention(x, encoder_output, encoder_output, src_mask)
→ Query: x（Decoder 側の表現）
→ Key/Value: encoder_output（Encoder 側の表現）
→ これにより、Decoder は「入力文のどの部分に注目すべきか」を学習します。
src_mask
→ 入力側のパディング部分を無視するためのマスクです。
x + self.dropout2(cross_attn_output)
→ 残差接続により、 元の Decoder 表現＋Encoder からの情報を統合します。
self.norm2(...)
→ LayerNorm で分布を安定化させます。
🐬 Add & Norm の役割まとめ
Add（残差接続）

入力 x とサブレイヤー出力 Sublayer(x) を足し合わせます。
数式: LayerNorm(x + Sublayer(x))
効果:
勾配が直接流れるため、深いネットワークでも学習が安定
情報の「ベースライン」を保ちつつ、差分だけを学習
Norm（Layer Normalization）

各サンプルの特徴ベクトルに対して、平均 0・分散 1 に正規化します。
効果:
学習率の設定がしやすくなる
勾配爆発・消失を防ぐ
バッチサイズに依存しない（BatchNorm と異なる）
🐳 まとめ
Decoder の Multi-Head Attention は 2 種類あり、
自己注意（Masked）
クロス注意（Encoder-Decoder）
で構成されています。
それぞれの後に Add & Norm が続き、
残差接続で勾配を流しやすくし、
LayerNorm で分布を安定させます。
この「Multi-Head Attention → Add & Norm」のブロックが N=6 層積み重なることで、
Transformer の Decoder は強力なシーケンス生成能力を獲得



---
Decoder の Feed-Forward Network（FFN） と、その後の Add & Norm について、
コードと詳細なコメントでご説明いたします。

🐟 Feed-Forward Network の役割
Transformer の各 Encoder/Decoder Layer には、
Multi-Head Attention の後に Feed-Forward Network が続きます。
役割:
Attention で得られた表現を、 非線形変換でさらに洗練する
各位置（トークン）ごとに独立して適用される（Position-wise）
論文の式（2）:

FFN
(
x
)
=
max
⁡
(
0
,
x
W
1
+
b
1
)
W
2
+
b
2
FFN(x)=max(0,xW
1
​
 +b
1
​
 )W
2
​
 +b
2
​

🐠 Feed-Forward Network の実装（コード＋コメント）


In [ ]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model, d_ff):
        """
        Feed-Forward Network の初期化
        - d_model: 入力・出力次元（例: 512）
        - d_ff: 中間層の次元（例: 2048）
        """
        super(FeedForwardNetwork, self).__init__()

        # ① 第1層の線形変換（d_model → d_ff）
        self.linear1 = nn.Linear(d_model, d_ff)

        # ② 第2層の線形変換（d_ff → d_model）
        self.linear2 = nn.Linear(d_ff, d_model)

        # ③ Dropout（論文 5.4 節の Residual Dropout）
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        """
        x: (batch_size, seq_len, d_model)
        """
        # ④ 第1層: 線形変換 + ReLU
        # 形状: (batch_size, seq_len, d_ff)
        x = self.linear1(x)
        x = F.relu(x)

        # ⑤ Dropout
        x = self.dropout(x)

        # ⑥ 第2層: 線形変換（d_ff → d_model）
        # 形状: (batch_size, seq_len, d_model)
        x = self.linear2(x)

        return x

コメント解説

d_model（例: 512） → d_ff（例: 2048） → d_model（例: 512）
→ ボトルネック構造ではなく、 一度広げてから戻す構造です。
ReLU により非線形性を導入し、表現力を高めます。
Dropout により過学習を防ぎます。
🐡 Decoder での Feed-Forward + Add & Norm
Decoder Layer 全体の流れ（コード＋コメント）:

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super(DecoderLayer, self).__init__()

        # ① Masked Multi-Head Attention + Add & Norm
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(0.1)

        # ② Encoder-Decoder Multi-Head Attention + Add & Norm
        self.cross_attention = MultiHeadAttention(d_model, num_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(0.1)

        # ③ Feed-Forward Network + Add & Norm
        self.feed_forward = FeedForwardNetwork(d_model, d_ff)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout3 = nn.Dropout(0.1)

    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        # Masked Self-Attention + Add & Norm
        attn_output, _ = self.self_attention(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout1(attn_output))

        # Encoder-Decoder Attention + Add & Norm
        cross_attn_output, _ = self.cross_attention(
            x, encoder_output, encoder_output, src_mask
        )
        x = self.norm2(x + self.dropout2(cross_attn_output))

        # ④ Feed-Forward + Add & Norm
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout3(ff_output))

        return x

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super(DecoderLayer, self).__init__()

        # ① Masked Multi-Head Attention + Add & Norm
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(0.1)

        # ② Encoder-Decoder Multi-Head Attention + Add & Norm
        self.cross_attention = MultiHeadAttention(d_model, num_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(0.1)

        # ③ Feed-Forward Network + Add & Norm
        self.feed_forward = FeedForwardNetwork(d_model, d_ff)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout3 = nn.Dropout(0.1)

    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        # Masked Self-Attention + Add & Norm
        attn_output, _ = self.self_attention(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout1(attn_output))

        # Encoder-Decoder Attention + Add & Norm
        cross_attn_output, _ = self.cross_attention(
            x, encoder_output, encoder_output, src_mask
        )
        x = self.norm2(x + self.dropout2(cross_attn_output))

        # ④ Feed-Forward + Add & Norm
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout3(ff_output))

        return x

コメント解説（Feed-Forward 部分）

ff_output = self.feed_forward(x)
→ Attention で得られた表現を、非線形変換でさらに洗練します。
x + self.dropout3(ff_output)
→ 残差接続により、元の表現に FFN の差分を加えます。
self.norm3(...)
→ LayerNorm で分布を安定化させます。
🐬 なぜ Feed-Forward が必要なのか？
Attention だけでは表現が線形に偏る

Attention は基本的に線形変換の組み合わせです。
FFN の ReLU により、 非線形な表現変換が可能になります。
各位置ごとに独立して適用される（Position-wise）

各トークンごとに独立して FFN を適用するため、
位置ごとの特徴抽出が可能です。
表現空間の拡張と圧縮

d_model → d_ff → d_model という流れで、
一度広い空間で変換してから元に戻すことで、
よりリッチな表現を獲得できます。
🐳 まとめ
Feed-Forward Network は、
Attention の後に続く非線形変換層
各位置ごとに独立して適用される
という特徴を持ちます。
Decoder では、
Masked Self-Attention → Add & Norm
Encoder-Decoder Attention → Add & Norm
Feed-Forward → Add & Norm
という 3 ステップが 1 つの Layer を構成し、それが N=6 層積み重なります。
この「Attention → FFN」の組み合わせが、Transformer の表現力を支える重要な要素

「output(shifted right)をインプットする」とは、
Transformer の Decoder が自己回帰的（auto-regressive）に動作するための仕組みを指しています。

以下、コードと詳細なコメントでご説明いたします。

🐟 「Outputs (shifted right)」の意味
Figure 1 の Decoder 側の入力ラベルに
Outputs (shifted right) と書かれています。

これは、

訓練時には、正解ラベルのシーケンスを 1 トークン右にずらしたものを Decoder の入力とする
推論時には、 自分がこれまで生成したトークンを入力とする
という意味です。

🐠 訓練時の「shifted right」の具体例
元の正解ラベル（ターゲット）

In [ ]:
["<s>", "I", "love", "you", "</s>"]

Decoder への入力（shifted right）

In [ ]:
["<s>", "I", "love", "you"]

Decoder が予測すべき出力

In [ ]:
["I", "love", "you", "</s>"]

In [ ]:
ポイント

入力は「1 トークン右にずらした正解ラベル」
出力は「元の正解ラベル（先頭の  <s>  を除く）」
これにより、Decoder は自分がこれまで生成したトークンだけを使って、次のトークンを予測する訓練ができます。
🐡 コードでの実装イメージ

In [ ]:
def prepare_decoder_inputs(target):
    """
    target: 正解ラベルのシーケンス (batch_size, tgt_len)
    """
    # 例: target = [[<s>, I, love, you, </s>]]

    # ① 最後のトークン（</s>）を削除
    decoder_input = target[:, :-1]  # [[<s>, I, love, you]]

    # ② 予測対象（ラベル）は 1 トークンずらしたもの
    decoder_target = target[:, 1:]  # [[I, love, you, </s>]]

    return decoder_input, decoder_target

訓練時の流れ

In [ ]:
# 入力シーケンス
src = ...  # (batch_size, src_len)

# 正解ラベル（ターゲット）
tgt = ...  # (batch_size, tgt_len) 例: [[<s>, I, love, you, </s>]]

# Decoder への入力（shifted right）
decoder_input, decoder_target = prepare_decoder_inputs(tgt)

# Transformer の順伝播
output = model(src, decoder_input)  # (batch_size, tgt_len-1, vocab_size)

# 損失計算（decoder_target と比較）
loss = criterion(output.view(-1, vocab_size), decoder_target.view(-1))

ポイント

入力は「1 トークン右にずらした正解ラベル」
出力は「元の正解ラベル（先頭の  ＜ｓ＞ を除く）」
これにより、Decoder は自分がこれまで生成したトークンだけを使って、次のトークンを予測する訓練ができます。
🐡 コードでの実装イメージ

🐬 なぜ「shifted right」が必要なのか？
1. 自己回帰性（Auto-regressive）の確保
Transformer の Decoder は、過去の出力のみを使って次のトークンを予測する必要があります。
訓練時に「正解ラベルをそのまま入力」してしまうと、
Decoder は未来の情報（自分がまだ生成していないトークン） を見てしまうことになります。
これを防ぐために、1 トークン右にずらした入力を使います。
2. 推論時の動作と一致させるため
推論時には、Decoder は以下のように動作します。
＜ｓ＞を入力 → 最初のトークン（例: "I"）を予測
＜ｓ＞, I を入力 → 次のトークン（例: "love"）を予測
＜ｓ＞, I, love を入力 → 次のトークン（例: "you"）を予測
…（以下同様）
訓練時の「shifted right」は、この推論時の動作を模倣しています。
🐳 Masked Self-Attention との関係
「shifted right」による入力設計と、
Masked Self-Attention（未来を見ないマスク） はセットで機能します。
訓練時:
入力: [＜ｓ＞, I, love, you]
マスク: 未来のトークン（自分がまだ生成していない部分）を見ない
予測: [I, love, you, ＜ｓ＞]
これにより、Decoder は正しく自己回帰的な振る舞いを学習できます。
🐋 まとめ
「Outputs (shifted right)」とは、
訓練時に正解ラベルを 1 トークン右にずらしたものを Decoder の入力とし、
推論時の「過去の出力のみを使う」動作を模倣する仕組みです。
これにより、Transformer の Decoder は自己回帰的なシーケンス生成を正しく学習



---

「Encoder は Attention → FFN なのに、Decoder は Attention → Attention → FFN なのはなぜ？」
という点について、Transformer の設計思想に沿ってご説明いたします。

🐟 Encoder と Decoder の構造の違い
Encoder（左側）
入力: ソース文（例: 英語の文）
役割: 入力文の内部表現を構築する
構造:
Self-Attention（入力文内の関係を学習）
→ Feed Forward（非線形変換で表現を洗練）
Decoder（右側）
入力: ターゲット文（shifted right）（例: ドイツ語の文）
役割: 入力文の情報を参照しつつ、出力文を生成する
構造:
Masked Self-Attention（出力文内の関係を学習）
→ Encoder-Decoder Attention（入力文の情報を参照）
→ Feed Forward（非線形変換で表現を洗練）
🐠 なぜ Decoder に「2 つの Attention」があるのか？
1. Masked Self-Attention（自己注意）の役割

In [ ]:
# Decoder Layer の一部
attn_output, _ = self.self_attention(x, x, x, tgt_mask)
x = self.norm1(x + self.dropout1(attn_output))

目的: 出力文（ターゲット）の内部構造を学習する
例:
「I love you」という文の中で、
「love」が「I」と「you」のどちらに強く関連するか、など
ただし、 未来を見ないマスクをかけるため、
時刻
i
i では
i
i より前のトークンしか参照できません。
2. Encoder-Decoder Attention（クロス注意）の役割

In [ ]:
# Decoder Layer の一部
cross_attn_output, _ = self.cross_attention(
    x, encoder_output, encoder_output, src_mask
)
x = self.norm2(x + self.dropout2(cross_attn_output))

目的: 入力文（ソース）の情報を参照しながら出力を生成する
Query: Decoder の表現（出力文側）
Key/Value: Encoder の最終出力（入力文側）
例:
英語「I love you」→ ドイツ語「Ich liebe dich」
Decoder が「liebe」を生成するとき、
Encoder の「love」に強く注目する、など
3. Feed Forward の役割

In [ ]:
# Decoder Layer の一部
ff_output = self.feed_forward(x)
x = self.norm3(x + self.dropout3(ff_output))

目的: Attention で得られた表現を非線形変換でさらに洗練する
Encoder/Decoder 共通の役割です。
🐡 なぜ Encoder には「2 つ目の Attention」がないのか？
Encoder は入力文だけを扱います。
そのため、
Self-Attention（入力文内の関係）
→ Feed Forward（表現の洗練）
だけで十分です。
Decoder は入力文と出力文の両方を扱うため、
自己注意（出力文内の関係）
クロス注意（入力文の参照）
の 2 つが必要になります。
🐬 まとめ
Encoder
入力文の内部表現を構築するだけなので、
Self-Attention → Feed Forward で十分。
Decoder
出力文の内部構造を学習（Masked Self-Attention）
入力文の情報を参照（Encoder-Decoder Attention）
表現を洗練（Feed Forward）
という 3 段階が必要。
このように、Decoder は「自分自身の文脈」と「入力文の文脈」の両方を扱うため、
Attention が 2 つ必要になっている、という設計



---
Transformer の Mask について、
特に Decoder の「未来を見ないマスク（Look-Ahead Mask）」と、
パディング用のマスク（Padding Mask）を、コードと詳細なコメントでご説明いたします。

🐟 Mask の種類
Transformer では、主に 2 種類のマスクが使われます。

Padding Mask（パディング用マスク）

入力シーケンスのパディング部分（無意味な部分） を無視するため
Encoder/Decoder の両方で使用
Look-Ahead Mask（未来を見ないマスク）

Decoder の Self-Attention で、 未来のトークンを見ないようにするため
自己回帰性（Auto-regressive）を保つために必要
🐠 1. Padding Mask（パディング用マスク）
コード＋詳細コメント


In [ ]:
def create_padding_mask(seq):
    """
    パディング部分を無視するためのマスクを生成
    - seq: 入力シーケンス（単語 ID の列） (batch_size, seq_len)
    """
    # 例: seq = [[1, 2, 3, 0, 0]]  （0 がパディング）

    # ① パディングでない位置を 1、パディングを 0 とする
    # 形状: (batch_size, 1, 1, seq_len)
    mask = (seq != 0).unsqueeze(1).unsqueeze(2)

    return mask

# 使用例
src = torch.tensor([[1, 2, 3, 0, 0]])  # パディングあり
src_mask = create_padding_mask(src)

print("src:", src)
print("src_mask shape:", src_mask.shape)
print("src_mask:", src_mask)
# 出力例:
# src: tensor([[1, 2, 3, 0, 0]])
# src_mask shape: torch.Size([1, 1, 1, 5])
# src_mask: tensor([[[[True, True, True, False, False]]]])

コメント解説

seq != 0
→ パディング（0）でない位置は True、パディングは False。
unsqueeze(1).unsqueeze(2)
→ 形状を (batch_size, 1, 1, seq_len) に変換（Multi-Head Attention 用）。
Attention 計算時に、mask == 0 の部分を -1e9 に置き換え、
softmax の重みをほぼ 0 にします。
🐡 2. Look-Ahead Mask（未来を見ないマスク）
コード＋詳細コメント

In [ ]:
def create_look_ahead_mask(size):
    """
    未来を見ないためのマスク（Look-Ahead Mask）を生成
    - size: シーケンス長
    """
    # ① 上三角行列（未来）を 1、それ以外を 0 にする
    # 例: size=4
    # [[0, 1, 1, 1],
    #  [0, 0, 1, 1],
    #  [0, 0, 0, 1],
    #  [0, 0, 0, 0]]
    mask = torch.triu(torch.ones(size, size), diagonal=1)

    # ② 未来を 0（見えない）、過去を 1（見える）に反転
    # [[1, 0, 0, 0],
    #  [1, 1, 0, 0],
    #  [1, 1, 1, 0],
    #  [1, 1, 1, 1]]
    return mask == 0

# 使用例
tgt_len = 4
look_ahead_mask = create_look_ahead_mask(tgt_len)

print("look_ahead_mask:")
print(look_ahead_mask)
# 出力例:
# tensor([[ True, False, False, False],
#         [ True,  True, False, False],
#         [ True,  True,  True, False],
#         [ True,  True,  True,  True]])

コメント解説

torch.triu(..., diagonal=1)
→ 上三角部分（未来）を 1、それ以外を 0 にします。
mask == 0
→ 未来を False（見えない）、過去を True（見える）に反転します。
Decoder の Self-Attention でこのマスクを使うと、
時刻
i
i では、
i
i より前のトークンしか参照できません。
これにより、自己回帰性が保たれます。
🐬 3. Decoder でのマスクの組み合わせ
Decoder では、Padding Mask と Look-Ahead Mask を組み合わせます。

In [ ]:
# ターゲットシーケンス（shifted right）
tgt = torch.tensor([[1, 2, 3, 0, 0]])  # パディングあり

# Padding Mask
padding_mask = create_padding_mask(tgt)  # (batch_size, 1, 1, tgt_len)

# Look-Ahead Mask
tgt_len = tgt.size(1)
look_ahead_mask = create_look_ahead_mask(tgt_len)  # (tgt_len, tgt_len)

# 両方のマスクを AND で結合
# 形状: (batch_size, 1, tgt_len, tgt_len)
tgt_mask = padding_mask & look_ahead_mask.unsqueeze(0).unsqueeze(0)

print("tgt_mask shape:", tgt_mask.shape)
print("tgt_mask:")
print(tgt_mask.squeeze())
# 出力例:
# tgt_mask shape: torch.Size([1, 1, 5, 5])
# tgt_mask:
# tensor([[ True, False, False, False, False],
#         [ True,  True, False, False, False],
#         [ True,  True,  True, False, False],
#         [ True,  True,  True, False, False],
#         [ True,  True,  True, False, False]])

コメント解説

padding_mask
→ パディング部分を無視（False）
look_ahead_mask
→ 未来のトークンを無視（False）
tgt_mask = padding_mask & look_ahead_mask
→ 両方の条件を満たす位置だけ True（見える）
🐳 4. Attention 計算でのマスクの使い方

In [ ]:
def scaled_dot_product_attention(query, key, value, mask=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        # mask == 0 の部分を -1e9（ほぼ -∞）に置き換え
        scores = scores.masked_fill(mask == 0, -1e9)

    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, value)

    return output, attention_weights

コメント解説

mask == 0 の部分を -1e9 にすることで、
softmax の入力が極端に小さくなり、重みがほぼ 0 になります。
これにより、
パディング部分への注意
未来のトークンへの注意
を防ぎます。
🐋 まとめ
Padding Mask
→ パディング部分を無視するため
Look-Ahead Mask
→ Decoder が未来を見ないようにするため（自己回帰性の確保）
両方を組み合わせることで、
Transformer は効率的かつ正しくシーケンスを処理



---
🐟 Q/K/V の直感的なイメージ
Attention は、「情報の検索システム」 と考えると分かりやすいです。

Query（クエリ）: 「何を知りたいか」という質問
Key（キー）: 情報の索引（インデックス）
Value（バリュー）: 実際の中身（情報そのもの）
例: 図書館で本を探す

Query: 「機械学習の入門書が欲しい」
Key: 本のタイトルやカテゴリ（索引）
Value: 本の内容（本文）
Attention は、Query と Key の類似度を計算し、
それに応じて Value を重み付きで集約します。

🐠 Scaled Dot-Product Attention での Q/K/V
コード＋コメント


In [ ]:
def scaled_dot_product_attention(query, key, value, mask=None):
    """
    query, key, value: (batch_size, seq_len, d_k)
    """
    d_k = query.size(-1)

    # ① Query と Key の類似度を計算（内積）
    # 形状: (batch_size, seq_len, seq_len)
    scores = torch.matmul(query, key.transpose(-2, -1))

    # ② スケーリング（1 / sqrt(d_k)）
    scores = scores / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    # ③ Softmax で重みを正規化
    attention_weights = F.softmax(scores, dim=-1)

    # ④ Value を重み付きで集約
    output = torch.matmul(attention_weights, value)

    return output, attention_weights

コメント解説

scores = Q @ K^T
→ Query と Key の類似度を計算します。
attention_weights = softmax(scores)
→ 類似度を確率分布（重み） に変換します。
output = weights @ V
→ Value を重み付きで足し合わせ、新しい表現を生成します。
🐡 Query / Key / Value の役割（具体例）
例: 英語 → 日本語翻訳
入力（ソース）: "I love you"
出力（ターゲット）: "私は あなたを 愛して います"
1. Encoder の Self-Attention（入力文内）
Query: 「love」という単語の表現（何を知りたいか？）
Key: 各単語の索引（"I", "love", "you"）
Value: 各単語の実際の意味
Attention の結果:

Query("love") は、Key("I") と Key("you") に強く関連する
→ Value("I") と Value("you") を多く取り入れた表現が生成される
2. Decoder の Encoder-Decoder Attention（クロス注意）
Query: Decoder 側の単語（例: 「愛して」）
Key: Encoder 側の各単語（"I", "love", "you"）
Value: Encoder 側の各単語の意味
Attention の結果:

Query("愛して") は、Key("love") に強く関連する
→ Value("love") を多く取り入れた表現が生成される
🐬 Q/K/V の数式的な意味
Attention の式（論文より）:

Attention
(
Q
,
K
,
V
)
=
softmax
(
Q
K
T
d
k
)
V
Attention(Q,K,V)=softmax(
d
k
​

​

QK
T

​
 )V
Q
K
T
QK
T
 : Query と Key の類似度行列
softmax: 類似度を確率分布に変換
V
V: 実際の情報を重み付きで集約
🐳 Multi-Head Attention での Q/K/V

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # 各 Head 用の線形変換
        self.w_q = nn.Linear(d_model, d_model)  # Query
        self.w_k = nn.Linear(d_model, d_model)  # Key
        self.w_v = nn.Linear(d_model, d_model)  # Value
        self.w_o = nn.Linear(d_model, d_model)  # 出力

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # 線形変換
        query = self.w_q(query)
        key = self.w_k(key)
        value = self.w_v(value)

        # Head ごとに分割
        query = query.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        key = key.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        value = value.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # 各 Head で Attention
        x, _ = scaled_dot_product_attention(query, key, value, mask)

        # Head を結合
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        # 最終線形変換
        output = self.w_o(x)

        return output, _

コメント解説

w_q, w_k, w_v は、各 Head が異なる表現空間を学習するための変換です。
例えば:
Head 1: 文法構造に注目する Q/K/V
Head 2: 共参照（anaphora）に注目する Q/K/V
Head 3: 長距離依存に注目する Q/K/V
これにより、 多角的な情報の検索が可能になります。
🐋 まとめ
Query: 「何を知りたいか」という質問（検索クエリ）
Key: 情報の索引（どの情報が関連するか）
Value: 実際の中身（情報そのもの）
Attention は、Query と Key の類似度に基づいて Value を重み付きで集約し、
文脈に応じた新しい表現を生成します。
この「Q/K/V」の考え方は、Transformer のすべての Attention 層（Self/Cross） に共通する基本概念



---
🐟 文法構造は「関係性」として表現される
Transformer は、文法構造を直接「ルール」として記述するのではなく、
以下のようにベクトル間の関係性として学習します。

単語（トークン）は、高次元ベクトル（埋め込み） として表現されます。
Attention は、どの単語がどの単語に強く関連するかを数値（重み）で表現します。
この「関連の強さ」が、 文法構造（主語・述語・修飾関係など）に対応します。
🐠 例：主語と動詞の関係
文: "The cat sleeps on the mat."
主語: "The cat"
動詞: "sleeps"
場所: "on the mat"
Transformer の Self-Attention では、以下のように学習されます。


In [ ]:
# 簡略化した Attention のイメージ
attention_weights = softmax(Q @ K^T / sqrt(d_k))

# "sleeps" に対する Attention 重みの例
# sleeps -> The: 0.1
# sleeps -> cat: 0.6   ← 主語に強く関連
# sleeps -> on: 0.1
# sleeps -> the: 0.1
# sleeps -> mat: 0.1

"sleeps"（動詞）は、"cat"（主語）に高い重みを割り当てます。
これが「主語-動詞の関係」の数値化です。
🐡 なぜベクトルで文法が表現できるのか？
1. 埋め込みベクトルが「意味＋文法情報」を含む
各単語は、例えば 512 次元のベクトルとして表現されます。
このベクトルには、
意味情報（猫、寝る、マットなど）
文法情報（名詞、動詞、前置詞など）
が混ざって埋め込まれます。
2. Attention が「関係性」を計算する
Query（クエリ）ベクトルと Key（キー）ベクトルの内積を計算します。
内積が大きいほど、「関連が強い」と解釈されます。
これにより、
主語と動詞
修飾語と被修飾語
代名詞とその参照先
などの文法関係が、数値（重み）として表現されます。
🐬 論文の例：Figure 3–5 の Attention 可視化
論文の付録（Attention Visualizations）では、
実際に Attention が文法構造を捉えている様子が示されています。

Figure 3:
動詞 "making" が、遠く離れた "more difficult" に強く注目
→ 「making ... more difficult」という句構造を捉えている
Figure 4:
"its" が、前の文のどの名詞を指しているか（共参照）を捉えている
Figure 5:
ある Head は文の構造（句・節） に注目している
これらはすべて、Attention の重み（数値） として可視化されています。

🐳 コードでのイメージ

In [ ]:
# 簡略化した例
# x: (batch_size, seq_len, d_model)
Q = x @ W_q  # Query
K = x @ W_k  # Key
V = x @ W_v  # Value

scores = Q @ K.transpose(-2, -1) / sqrt(d_k)
attention_weights = softmax(scores)  # 文法関係がここに現れる
output = attention_weights @ V

attention_weights が、単語間の文法関係を数値で表しています。
学習を通じて、W_q, W_k, W_v が文法構造を捉えるように調整されます。
🐋 まとめ
Transformer は、文法構造を直接ルールとして記述するのではなく、
単語をベクトルで表現し、
Attention で「どの単語がどの単語に強く関連するか」を数値（重み）で学習します。
これにより、 主語-動詞関係、修飾関係、共参照などの文法構造が、
ベクトル間の関係性として数値化されます。
この「関係性の数値化」が、Transformer が文法構造を扱える理由



---


🐟 共参照の直感的な意味
共参照とは、文中の異なる表現が同じ実体を指している関係のことです。

例:

"John has a dog. He loves it."
"He" → "John"（同じ人物）
"it" → "dog"（同じ犬）
このように、

代名詞（he, she, it, they など）
指示詞（this, that など）
別の名詞句
が、同じもの（実体）を指していることを「共参照」と呼びます。

🐠 Transformer における共参照の扱い
Transformer は、Self-Attention を通じて共参照を学習します。

例: "The cat sleeps on the mat. It is soft."
"It" は何を指しているか？
"cat" なのか？
"mat" なのか？
Transformer の Attention では、以下のように学習されます。

In [ ]:
# "It" に対する Attention 重みの例
# It -> The: 0.1
# It -> cat: 0.1
# It -> sleeps: 0.1
# It -> on: 0.1
# It -> the: 0.1
# It -> mat: 0.5   ← "mat" に強く関連

"It" は "mat" に高い重みを割り当てている
→ 「It は mat を指している」と解釈できます。
🐡 論文の例：Figure 4（共参照の Attention）
論文の Figure 4 では、共参照を捉える Attention Head が示されています。

文: "The law that was passed in 1997 was its first step."
"its" が何を指しているか？
"law" なのか？
"1997" なのか？
それとも別のものか？
Attention の可視化結果では、

ある Head が "its" → "law" に強く注目している
別の Head が "its" → "1997" に注目している
といった複数の解釈が示されています。

これが、共参照を数値（重み）として捉えている例です。

🐬 なぜ Attention で共参照が捉えられるのか？
Query / Key / Value の役割

Query("its"): 「何を指しているか？」という質問
Key(各単語): 候補となる単語の索引
Value(各単語): 実際の意味
類似度計算

Query と Key の内積により、 どの単語が「its」の指す対象としてふさわしいかを評価します。
学習を通じて、代名詞が正しい実体に高い重みを割り当てるように調整されます。
Multi-Head の効果

複数の Head が、 異なる共参照関係を学習します。
例:
Head 1: "its" → "law"
Head 2: "its" → "1997"
Head 3: 文全体の構造を見る

In [ ]:
# 簡略化した共参照の Attention
# x: (batch_size, seq_len, d_model)
Q = x  # Query（代名詞 "its" など）
K = x  # Key（文中のすべての単語）
V = x  # Value（実際の意味）

scores = Q @ K.transpose(-2, -1) / sqrt(d_k)
attention_weights = softmax(scores)  # 共参照関係がここに現れる
output = attention_weights @ V

attention_weights のうち、代名詞の行を見ると、
どの単語に高い重みがあるかが分かります。
これが共参照の数値化です。
🐋 まとめ
共参照とは、文中の異なる表現が同じ実体を指している関係です。
Transformer は、Self-Attention を通じて、
代名詞がどの名詞を指しているか
指示詞が何を指しているか
を数値（重み）として学習します。
論文の Figure 4 では、実際に Attention Head が共参照を捉えている様子が可視化されています。
このように、Transformer は文法構造だけでなく、共参照のような意味的な関係も数値化



---


長い文での文脈は数値化できるか？

---
はい、Transformer は長い文での文脈も数値化できます。
ただし、その方法と限界について、順を追ってご説明いたします。

🐟 文脈の数値化とは？
「文脈の数値化」とは、文全体の意味や構造を、ベクトルや重みとして表現することです。

単語ベクトル（埋め込み）
Attention 重み（どの単語がどの単語に注目するか）
層を重ねた後の隠れ状態
これらを通じて、長い文の文脈も数値として扱えます。

🐠 Self-Attention による文脈の数値化
1. 全単語間の関係を計算
Self-Attention は、すべての単語ペア間の関係を計算します。

入力: (batch_size, seq_len, d_model)
Attention 重み: (batch_size, seq_len, seq_len)
例:
"The cat sleeps on the mat because it is tired."

"tired" は、"cat" や "sleeps" に強く注目する
"because" は、原因と結果の関係を捉える
これらが数値（重み） として表現されます。

2. 多層化による文脈の深い理解
Transformer は N=6 層の Encoder/Decoder を持ちます。

第1層: 局所的な関係（隣接単語）
第2層: 少し広い範囲（句・節）
第3層以上: 文全体の構造・意味
層を重ねることで、文脈が徐々に深く・広く表現されます。

🐡 長い文での限界と対策
1. 計算量の増大
Self-Attention の計算量は O(seq_len²) です。

seq_len=100 → 10,000 ペア
seq_len=1,000 → 1,000,000 ペア
長すぎると、メモリ・計算時間が膨大になります。

2. 論文での対策（Positional Encoding と Mask）
Positional Encoding:
位置情報を sin/cos で埋め込み、 遠くの単語も相対位置として扱えるようにします。
Mask:
Padding Mask で無効な単語を無視し、Look-Ahead Mask で未来の単語を見ないようにします。
これにより、長い文でも位置関係を保ったまま数値化できます。

🐬 実際の長文での例
例: 長い説明文
"The company, which was founded in 1990 by two engineers who had previously worked at a large tech firm, decided to pivot its business model after the 2008 financial crisis, focusing on cloud services and AI solutions."

"which" → "company"（共参照）
"who" → "engineers"（共参照）
"after" → "2008 financial crisis"（時間関係）
"focusing on" → "cloud services and AI solutions"（目的）
これらが Attention 重みとして数値化されます。


In [ ]:
# 長い文での Self-Attention（簡略化）
# x: (batch_size, seq_len, d_model)
Q = x @ W_q
K = x @ W_k
V = x @ W_v

scores = Q @ K.transpose(-2, -1) / sqrt(d_k)
attention_weights = softmax(scores)  # 文脈関係がここに現れる
output = attention_weights @ V

attention_weights の各行が、各単語が文全体のどこに注目しているかを表します。
これが長い文の文脈の数値化です。
🐋 まとめ
Transformer は、Self-Attention と多層構造により、長い文の文脈も数値化できます。
計算量の増大という限界はありますが、Positional Encoding と Mask により、 位置関係を保ったまま文脈を扱うことが可能です。
実際の長文では、共参照・時間関係・因果関係などが Attention 重み



---

attention_weights について、Transformer の実装に即してご説明いたします。

🐟 attention_weights の定義
attention_weights は、Scaled Dot-Product Attention の計算過程で得られる
ソフトマックス（softmax）適用後の重み行列です。

コードで言うと、以下の部分です。


scores = Q @ K.transpose(-2, -1) / sqrt(d_k)
attention_weights = softmax(scores)  # ← これが attention_weights
output = attention_weights @ V
形状: (batch_size, num_heads, seq_len, seq_len)
（簡略化すると (seq_len, seq_len)）
🐠 何を表しているか？
attention_weights は、「各単語が、文中のどの単語にどれだけ注目しているか」 を数値で表します。

例: 文 "The cat sleeps on the mat."
行: 注目する側（Query）
列: 注目される側（Key）

# 簡略化した attention_weights の例（1ヘッド分）
#        The   cat  sleeps on   the  mat
weights = [
    [0.8, 0.1, 0.0, 0.0, 0.1, 0.0],  # The
    [0.1, 0.6, 0.2, 0.0, 0.1, 0.0],  # cat
    [0.0, 0.5, 0.3, 0.1, 0.0, 0.1],  # sleeps
    [0.0, 0.0, 0.1, 0.6, 0.2, 0.1],  # on
    [0.1, 0.0, 0.0, 0.2, 0.5, 0.2],  # the
    [0.0, 0.0, 0.1, 0.1, 0.2, 0.6]   # mat
]
3行目（"sleeps"）を見ると:
"cat" に 0.5、自身に 0.3、など
→ "sleeps" は "cat" に強く注目している（主語-動詞関係）
🐡 なぜ重要か？
attention_weights は、Transformer が文法構造や文脈を数値化した結果そのものです。

文法構造: 主語-動詞、修飾関係など
共参照: 代名詞がどの名詞を指しているか
文脈: 遠くの単語との関係
これらがすべて、0〜1の数値（重み） として表現されます。

🐬 論文での役割
論文の Figure 3–5 では、attention_weights を可視化しています。

Figure 3: "making ... more difficult" の関係
Figure 4: "its" が何を指しているか（共参照）
Figure 5: 文の構造（句・節）を捉える Head
これらはすべて、attention_weights の数値を色の濃淡で表現したものです。

🐳 まとめ
attention_weights は、softmax 適用後の重み行列です。
各行が「各単語が文中のどこに注目しているか」を数値で表します。
これにより、 文法構造・共参照・文脈が数値化



---
🐟 全体の流れ（イメージ）
Transformer の最後では、
「どの単語が出てきそうか？」を確率で表すために、以下の流れを使います。

linear（線形変換）: 点数をつける
softmax（ソフトマックス）: 点数を確率に変換
outprobabilities（出力確率）: 各単語の出る確率
🐠 ① linear（線形変換）: 点数をつける
何をしているか？
モデルが学習した情報をもとに、各単語に「点数（スコア）」をつけます。
点数が高いほど、「次にその単語が出てきそう」という意味です。
例: 次に来る単語を予測
文: "I have a ..."
候補: "cat", "dog", "book", "idea"


In [ ]:
# linear の出力（点数）の例
scores = {
    "cat": 8.2,
    "dog": 7.5,
    "book": 3.1,
    "idea": 2.0
}

"cat" が一番点数が高い → 一番出てきそう
🐡 ② softmax（ソフトマックス）: 点数を確率に変換
何をしているか？
点数を 0〜1 の確率 に変換します。
すべての確率を足すと 1（100%） になります。
計算のイメージ
点数を指数関数
e
x
e
x
  で大きくする
全体の合計で割る

In [ ]:
# 簡略化した計算（高校数学レベル）
exp_scores = {
    "cat": exp(8.2) ≈ 3640,
    "dog": exp(7.5) ≈ 1800,
    "book": exp(3.1) ≈ 22,
    "idea": exp(2.0) ≈ 7.4
}

total = 3640 + 1800 + 22 + 7.4 ≈ 5469.4

probabilities = {
    "cat": 3640 / 5469.4 ≈ 0.665,
    "dog": 1800 / 5469.4 ≈ 0.329,
    "book": 22 / 5469.4 ≈ 0.004,
    "idea": 7.4 / 5469.4 ≈ 0.001
}

"cat": 約 66.5%
"dog": 約 32.9%
"book": 約 0.4%
"idea": 約 0.1%
→ 確率として解釈できる形になります。

🐬 ③ outprobabilities（出力確率）: 確率の完成
何をしているか？
softmax の結果が、そのまま 出力確率（outprobabilities） です。
これを使って、
最も確率が高い単語を選ぶ（貪欲法）
確率に応じてランダムに選ぶ（サンプリング）
などの方法で、 次の単語を決めます。
例: 次の単語の決定
"I have a cat"（確率 66.5% で "cat" を選択）
🐳 なぜこの流れが必要か？
linear:
モデルが学習した知識を「点数」として表現します。
softmax:
点数を確率に変換し、 比較しやすい形にします。
outprobabilities:
確率として扱えるので、 次の単語の選択や評価がしやすくなります。
🐋 まとめ（高校生向け）
linear: 各単語に「出てきそう度」の点数をつける
softmax: 点数を 0〜1 の確率に変換（全部足して 100%）
outprobabilities: 完成した確率。これを使って次の単語を決める
この流れにより、Transformer は
「次にどの単語が来るか？」を確率として予測



---
左が二層構造、右が三層構造になっている理由についてご説明いたします。

🐟 図の構造の確認
論文の Figure 1 では、以下のように描かれています。

左側（Encoder）:

Self-Attention
Feed-Forward
→ 2つのサブ層（Add & Norm を含めると 2 ブロック）
右側（Decoder）:

Masked Self-Attention
Encoder-Decoder Attention
Feed-Forward
→ 3つのサブ層（Add & Norm を含めると 3 ブロック）
🐠 なぜ Encoder は 2 層、Decoder は 3 層なのか？
1. Encoder の役割
入力文を理解する（ソース文の意味・構造を捉える）
必要な処理:
Self-Attention: 文中の単語同士の関係を捉える
Feed-Forward: 各位置の表現を非線形に変換（知識の統合）
→ これで十分に文の意味表現が作れます。

2. Decoder の役割
出力文を生成する（ターゲット文を順に作る）
必要な処理:
Masked Self-Attention:
過去の単語だけを見て、次の単語を予測
未来の単語を見ないようにする（マスク）
Encoder-Decoder Attention:
Encoder が作った「入力文の意味表現」を参照
翻訳なら「どの単語に対応するか」を捉える
Feed-Forward:
最終的な単語の表現を非線形に変換
→ Decoder は、自分自身の文脈と入力文の文脈の両方を見る必要があるため、
1つ多い Attention 層が必要になります。

🐡 直感的な例
翻訳の例: "I love you" → "私はあなたを愛しています"
Encoder（左）:

"I", "love", "you" の関係を捉える
→ Self-Attention + Feed-Forward で十分
Decoder（右）:

Masked Self-Attention:
既に出した「私は」「あなたを」を見て、次を予測
Encoder-Decoder Attention:
Encoder の "love" に注目して、「愛しています」を生成
Feed-Forward:
最終的な単語表現を整える
→ Decoder は自分と入力の両方を見るので、 3層構造になります。

🐬 まとめ
Encoder（左）:
入力文の理解だけなので、 Self-Attention + Feed-Forward の 2 層で十分。
Decoder（右）:
自分自身の文脈（Masked Self-Attention）
入力文の文脈（Encoder-Decoder Attention）
最終的な表現（Feed-Forward）
の 3 層が必要。
この違いのため、図では左が二層、右が三層という構造




---
「評価」という言葉の使い方について、
Transformer の文脈に沿って整理してご説明いたします。

🐟 一般的な「評価」の意味
日常的には、「評価」は以下のような意味で使われます。

物事の価値や良し悪しを判断すること
成績や性能を測ること
レビューや批評をすること
例:

「このモデルの性能を評価する」
「テストで良い評価をもらった」
🐠 Transformer における「評価」の使い方
Transformer の文脈では、「評価」は主に2つの意味で使われます。

1. モデル全体の性能評価（Evaluation）
学習済みモデルがどれだけ良いか悪いかを測ること
例: BLEU スコア、精度、損失など
2. Attention における「類似度評価」（Scoring）
Query と Key の内積を計算し、どれだけ似ているか（関連が強いか）を数値で表すこと
これは「良し悪し」ではなく、 関係の強さを測る意味での評価です。


In [ ]:
scores = Q @ K.transpose(-2, -1) / sqrt(d_k)  # 類似度を「評価」

🐡 linear は「評価」なのか？
先ほどの linear について、

モデルを評価しているわけではない
モデルが作ったベクトルを、単語の点数に変換している
という点を踏まえると、linear は以下のように言い換えられます。

「モデルを線形評価している」→ 誤解を招く表現
「モデルが学習した情報を、線形変換で単語のスコアにまとめている」→ 正しい表現
つまり、linear は評価関数ではなく、変換関数です。

🐬 適切な言い換え例
「linear でモデルを評価する」→ ✗（誤解）
「linear で単語のスコアを計算する」→ ○
「Attention で Query と Key の類似度を評価する」→ ○（関係の強さを測る意味）
🐳 まとめ
「評価」は、
モデル全体の性能を測る場合
Attention で類似度を測る場合
に使われることが多いです。
linear は評価ではなく、線形変換です。
言葉の使い分けが重要で、
「評価」＝「良し悪しを判断」と捉えると誤解



---


🐟 softmax は何をするものか？

---


softmax は、「点数（スコア）を確率に変換する」 ための計算です。

入力: 各候補の点数（大きいほど良い）
出力: 各候補の確率（0〜1、全部足して1）
例:

テストの点数 → 順位や確率に変換するイメージ
🐠 具体例で見てみよう
例: 次に来る単語の予測
文: "I have a ..."
候補: "cat", "dog", "book", "idea"

モデルがつけた点数（linear の出力）:

In [ ]:
scores = {
    "cat": 8.2,
    "dog": 7.5,
    "book": 3.1,
    "idea": 2.0
}

このままでは、

どれが一番出てきそうか分かりにくい
確率として扱えない
→ softmax で確率に変換します。

🐡 softmax の計算ステップ（高校数学レベル）
1. 指数関数
e
x
e
x
  で点数を大きくする

In [ ]:
exp_scores = {
    "cat": exp(8.2) ≈ 3640,
    "dog": exp(7.5) ≈ 1800,
    "book": exp(3.1) ≈ 22,
    "idea": exp(2.0) ≈ 7.4
}

点数が高いほど、指数関数でとても大きくなるのがポイントです。
2. 全部足し算する

In [ ]:
total = 3640 + 1800 + 22 + 7.4 ≈ 5469.4

3. それぞれを合計で割る（確率にする）

In [ ]:
probabilities = {
    "cat": 3640 / 5469.4 ≈ 0.665,
    "dog": 1800 / 5469.4 ≈ 0.329,
    "book": 22 / 5469.4 ≈ 0.004,
    "idea": 7.4 / 5469.4 ≈ 0.001
}

これが softmax の出力（確率） です。

"cat": 約 66.5%
"dog": 約 32.9%
"book": 約 0.4%
"idea": 約 0.1%
→ 全部足すと 100% になります。

🐬 なぜ softmax を使うのか？
確率として扱える

0〜1 の値で、全部足して 1
「どの単語がどれだけ出てきそうか」が分かる
点数差を強調できる

8.2 と 7.5 は少しの差ですが、
softmax 後は 66.5% と 32.9% で大きく差がつく
モデルが「これだ！」と判断した候補が際立つ
学習（誤差逆伝播）に適している

微分が簡単で、機械学習の計算に都合が良い
🐳 数式（参考）
softmax の数式は以下の通りです。

softmax
(
x
i
)
=
e
x
i
∑
j
e
x
j
softmax(x
i
​
 )=
∑
j
​
 e
x
j
​


e
x
i
​


​

x
i
x
i
​
 : i番目の候補の点数
e
x
i
e
x
i
​

 : 指数関数
分母: すべての候補の指数の合計
🐋 まとめ（高校生向け）
softmax は、点数を確率に変える計算です。
手順:
点数を指数関数で大きくする
全部足す
それぞれを合計で割る
これで、「どの単語がどれだけ出てきそうか」が確率で分かるよ



---

linear の数式とコードについて、Transformer の文脈に沿ってご説明いたします。

🐟 linear の数式
Transformer の最後にある linear は、線形変換（アフィン変換） です。

数式:

linear
(
x
)
=
x
W
+
b
linear(x)=xW+b
x
x: 入力ベクトル（隠れ状態、形状:
d
model
d
model
​
 ）
W
W: 重み行列（形状:
d
model
×
vocab_size
d
model
​
 ×vocab_size）
b
b: バイアスベクトル（形状:
vocab_size
vocab_size）
出力: 各単語のスコア（形状:
vocab_size
vocab_size）
🐠 linear の役割
入力: モデルが作った最終的なベクトル（文脈を理解した結果）
出力: 各単語の「次に来そう度」の点数（スコア）
例:

入力次元:
d
model
=
512
d
model
​
 =512
語彙数:
vocab_size
=
30000
vocab_size=30000
出力: 30000 個のスコア（各単語に1つ）
🐡 PyTorch でのコード例
1. 単純な linear 層の定義

In [ ]:
import torch
import torch.nn as nn

# パラメータ
d_model = 512
vocab_size = 30000

# linear 層の定義
linear_layer = nn.Linear(d_model, vocab_size)

# 入力の例 (batch_size=1, seq_len=1, d_model=512)
x = torch.randn(1, 1, d_model)  # モデルが作った最終ベクトル

# linear の計算
scores = linear_layer(x)  # 形状: (1, 1, vocab_size)

print("入力形状:", x.shape)
print("出力形状:", scores.shape)
print("スコアの例（最初の5単語）:", scores[0, 0, :5])

2. 数式との対応
nn.Linear は内部的に
x
W
+
b
xW+b を計算します。
W: linear_layer.weight（形状: (vocab_size, d_model)）
b: linear_layer.bias（形状: (vocab_size,)）

In [ ]:
# 手動で計算（確認用）
W = linear_layer.weight  # (vocab_size, d_model)
b = linear_layer.bias     # (vocab_size,)

# x: (1, 1, d_model) -> (1, 1, d_model)
# W^T: (d_model, vocab_size)
# 計算: x @ W^T + b
manual_scores = x @ W.T + b  # 形状: (1, 1, vocab_size)

# 結果が一致するか確認
print("一致するか:", torch.allclose(scores, manual_scores))

🐬 Transformer 全体での linear の位置
Transformer の最後では、以下の流れになります。

In [ ]:
# 最終的な隠れ状態 (batch_size, seq_len, d_model)
hidden_states = ...  # Decoder の出力

# linear でスコアに変換
scores = linear_layer(hidden_states)  # (batch_size, seq_len, vocab_size)

# softmax で確率に変換
probabilities = torch.softmax(scores, dim=-1)  # (batch_size, seq_len, vocab_size)

In [ ]:
scores: 各単語の点数
probabilities: 各単語の確率（outprobabilities）
🐳 まとめ
数式:
linear
(
x
)
=
x
W
+
b
linear(x)=xW+b
役割: モデルが作ったベクトルを、単語のスコアに変換する
コード: nn.Linear(d_model, vocab_size) で実装
出力: 各単語の「次に来そう度」の点数（softmax の入力）
もし、nn.Linear の内部動作（重みの初期化や勾配計算など）について、さらに詳しい解説が必要でしたら、お気軽にお申し付けください。

input に対する output の例について、
Transformer の翻訳タスクを想定して、具体的にご説明いたします。

🐟 全体の流れ（イメージ）
input: ソース文（例: 英語）
output: ターゲット文（例: 日本語）
Transformer は、
input を一度に受け取り、output を一度に生成します（訓練時）。

🐠 具体例（英語 → 日本語）
input（英語）

"I have a cat."
output（日本語）

「私は猫を飼っています。」
Transformer の内部では、以下のように処理されます。

🐡 Encoder 側の input/output
Encoder input
トークン列（例）: ["I", "have", "a", "cat", "."]
埋め込み後: (batch_size=1, seq_len=5, d_model=512)
Encoder output
各位置の文脈を理解したベクトル
形状: (batch_size=1, seq_len=5, d_model=512)
→ "I", "have", "a", "cat", "." それぞれが、
文全体の意味を考慮したベクトルになります。

🐬 Decoder 側の input/output
Decoder input（訓練時）
ターゲット文を1トークン右にずらしたもの（shifted right）
例:
正解: ["私は", "猫を", "飼っています", "。"]
input: [<sos>, "私は", "猫を", "飼っています"]
Decoder output（訓練時）
各位置で「次に来る単語の確率分布」を出力
例:
位置1: <sos> を見て → 「私は」の確率が高い
位置2: <sos> 私は を見て → 「猫を」の確率が高い
位置3: <sos> 私は 猫を を見て → 「飼っています」の確率が高い
位置4: <sos> 私は 猫を 飼っています を見て → 「。」の確率が高い
🐳 コードでのイメージ

In [ ]:
# 簡略化した例
# input_ids: (batch_size, src_len)
# target_ids: (batch_size, tgt_len)

# Encoder
encoder_output = encoder(input_ids)  # (batch_size, src_len, d_model)

# Decoder（訓練時）
decoder_output = decoder(
    input_ids=target_ids[:, :-1],  # shifted right（最後のトークンを除く）
    encoder_output=encoder_output
)  # (batch_size, tgt_len-1, vocab_size)

# 各位置の出力確率
probabilities = torch.softmax(decoder_output, dim=-1)
# 例: 位置1で「私は」の確率が 0.9、位置2で「猫を」の確率が 0.8、など

🐋 推論時（生成時）の input/output
推論時の流れ（貪欲法の例）
<sos> を input として、最初の単語を生成
生成した単語を input に追加し、次の単語を生成
<eos> が出るまで繰り返す
例:

input: <sos> → output: 「私は」
input: <sos> 私は → output: 「猫を」
input: <sos> 私は 猫を → output: 「飼っています」
input: <sos> 私は 猫を 飼っています → output: 「。」
終了
🐟 まとめ
input: ソース文（Encoder）＋ ターゲット文の過去部分（Decoder）
output: 各位置での「次に来る単語の確率分布」
訓練時は一度に処理、推論時は順次生成
このように、Transformer は
input に対して output を確率分布として出力し、
それを元に翻訳文を構築

Transformer は、翻訳や推論（文章生成）以外にも、非常に多くのタスクに応用できます。
代表的なものをご紹介いたします。

🐟 自然言語処理（NLP）タスク
1. テキスト分類
入力: 文や文章
出力: カテゴリやラベル
例:
感情分析（ポジティブ/ネガティブ）
スパム判定
ニュースのジャンル分類
2. 質問応答（QA）
入力: 文書 + 質問
出力: 答えの文やスパン
例:
SQuAD データセット（文書中の答えの位置を特定）
3. 要約
入力: 長い文書
出力: 短い要約文
例:
ニュース記事の要約
論文のアブストラクト生成
4. 固有表現認識（NER）
入力: 文
出力: 人名・組織名・地名などのタグ
例:
"Apple is a company." → "Apple" = ORGANIZATION
🐠 マルチモーダルタスク（テキスト＋他 modality）
1. 画像キャプション生成
入力: 画像
出力: 説明文
例:
画像 → "A cat sleeping on a mat."
2. 視覚質問応答（VQA）
入力: 画像 + 質問
出力: 答え
例:
画像 + "What is the cat doing?" → "sleeping"
3. 音声認識（ASR）
入力: 音声波形
出力: テキスト
例:
Whisper などのモデルで利用
🐡 コード生成・プログラム解析
1. コード生成
入力: 自然言語の説明
出力: プログラムコード
例:
"Pythonでクイックソートを実装して" → コード生成
2. コード翻訳
入力: ある言語のコード
出力: 別の言語のコード
例:
Python → JavaScript の変換
3. プログラム解析
入力: コード
出力: バグ検出、脆弱性検出、ドキュメント生成など
🐬 その他の応用
1. 推薦システム
入力: ユーザーの行動履歴やアイテム情報
出力: おすすめアイテムの確率
例:
動画推薦、商品推薦
2. 時系列予測
入力: 過去の時系列データ
出力: 未来の値
例:
株価予測、需要予測
3. 化学・生物情報学
入力: 分子構造（SMILES表記など）
出力: 物性予測、薬剤候補の生成
例:
タンパク質の構造予測（AlphaFold なども関連技術）
🐳 なぜこんなに多くのタスクができるのか？
Transformer の核心は、Self-Attention による「関係性の学習」 です。

テキスト: 単語間の関係（文法・意味）
画像: パッチ間の関係（物体・背景）
音声: フレーム間の関係（音素・単語）
コード: トークン間の関係（構文・依存関係）
この「関係性を数値化する能力」が、様々なデータ形式に通用するため、
多岐にわたるタスクで活躍できます。

🐋 まとめ
Transformer は、以下のような幅広いタスクに応用可能です。

自然言語処理（分類、QA、要約、NERなど）
マルチモーダル（画像キャプション、VQA、音声認識）
コード生成・解析
推薦システム、時系列予測、化学・生物情報学 など
「Attention Is All You Need」のタイトル通り、
Attention という仕組みが、多くのタスクの基盤となっている



---


Transformer は回帰（Regression） や分類（Classification） にも応用できます。
それぞれの方法についてご説明いたします。

🐟 分類タスク（Classification）
1. テキスト分類の例
入力: 文や文章（例: 商品レビュー）
出力: クラスラベル（例: ポジティブ/ネガティブ）
実装の流れ
Transformer（Encoder）で文を処理
最終層の出力から、特定のトークン（例: [CLS]）のベクトルを取り出す
そのベクトルを linear 層でクラス数に変換
softmax で確率に変換

In [ ]:
import torch
import torch.nn as nn

class TransformerClassifier(nn.Module):
    def __init__(self, transformer, num_classes):
        super().__init__()
        self.transformer = transformer
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, input_ids):
        # Transformer で文を処理
        outputs = self.transformer(input_ids)  # (batch_size, seq_len, d_model)

        # [CLS] トークンのベクトルを取り出す（先頭トークンなど）
        cls_output = outputs[:, 0, :]  # (batch_size, d_model)

        # 分類スコア
        logits = self.classifier(cls_output)  # (batch_size, num_classes)
        return logits

# 使用例
model = TransformerClassifier(transformer, num_classes=2)
logits = model(input_ids)
probabilities = torch.softmax(logits, dim=-1)

例: 感情分析、スパム判定、トピック分類など
🐠 回帰タスク（Regression）
1. 数値予測の例
入力: 時系列データやテキスト特徴量
出力: 連続値（例: 価格、スコア、需要など）
実装の流れ
Transformer で入力系列を処理
最終層の出力から、代表ベクトル（平均や [CLS]） を取り出す
linear 層で 1 次元（または多次元）の数値に変換

In [ ]:
class TransformerRegressor(nn.Module):
    def __init__(self, transformer):
        super().__init__()
        self.transformer = transformer
        self.regressor = nn.Linear(d_model, 1)  # 1次元の出力

    def forward(self, input_ids):
        outputs = self.transformer(input_ids)  # (batch_size, seq_len, d_model)

        # 系列全体の平均を取る（または [CLS] を使う）
        pooled_output = outputs.mean(dim=1)  # (batch_size, d_model)

        # 回帰値
        prediction = self.regressor(pooled_output)  # (batch_size, 1)
        return prediction

# 使用例
model = TransformerRegressor(transformer)
pred = model(input_ids)  # 例: 株価の予測値など

: 株価予測、需要予測、スコア予測など
🐡 なぜ Transformer で回帰・分類ができるのか？
1. Encoder の出力が「文の意味ベクトル」だから
Transformer の Encoder は、文全体の意味を考慮したベクトル列を出力します。
このベクトルを集約（平均や [CLS]）することで、 文レベルの特徴量を得られます。
2. linear 層で任意の次元に変換できる
nn.Linear を使うことで、
分類: クラス数次元
回帰: 1次元（または多次元）
に簡単に変換できます。
3. 事前学習済みモデルの利用
BERT や GPT などの事前学習済み Transformer を利用し、
最後の層だけ差し替える（ファインチューニング）
ことで、少量のデータでも高性能な分類・回帰が可能です。
🐬 具体例
分類の例: 感情分析
入力: "This movie is great!"
Transformer → [CLS] ベクトル → linear → softmax
出力: ポジティブ: 0.95, ネガティブ: 0.05
回帰の例: レビューの評価予測
入力: 商品レビューのテキスト
Transformer → 平均ベクトル → linear
出力: 評価スコア（例: 4.2/5.0）
🐳 まとめ
分類:
Transformer の出力ベクトルを linear → softmax でクラス確率に変換
回帰:
Transformer の出力ベクトルを linear で数値に変換
Transformer は、文の意味をベクトルで捉えることができるため、
その後の linear 層を変えるだけで、 多様なタスク（翻訳、生成、分類、回帰）に応用